# Knowledge Graphs for Data Engineers masterclass

This notebook walks through the **knowledge engineering** pipeline step by step. From raw data frames to a fully operational knowledge graph with ontology, SKOS vocabularies, DCAT metadata, FIBO alignment, inference rules, SHACL validation and SPARQL queries.

The data engineering is handled by `data_engineering.py`, which loads data from **four different source formats**:

| Source | Format | Why |
|--------|--------|-----|
| Customers | Parquet | Close to Databricks / lakehouse |
| Orders | CSV | Classic flat-file exchange |
| Products | Excel (.xlsx) | Typical business-managed source |
| Support Tickets | SQL (Polars SQLContext) | Simulating a SQL endpoint |

Each source uses **its own naming conventions**; the same customer foreign key is called `customer_id` in one file, `cust_ref` in another, and `client_id` in a third. A column called `name` means a person in one file and a product in another. The data engineering layer handles this **entity harmonization** before we ever touch the knowledge graph.

We import those functions and focus here on the **knowledge engineering** side: how to turn data frames into a knowledge graph, enrich it, reason over it, validate it, and query it.

## 0. Setup

We need to run from the project root so that all relative paths (`data/`, `tpl/`, `ttl/`, `queries/`) resolve correctly.

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), '') if os.path.basename(os.getcwd()) == 'notebook' else '.')
# Ensure we're in the project root
print("Working directory:", os.getcwd())

Working directory: /Users/veronahe/Documents/masterclasses/The Repo/data-engineering-to-knowledge-engineering/full-demo


In [2]:
import sys
sys.path.insert(0, os.getcwd())

from maplib import Model
import data_engineering as data
from utils import print_count

## 1. Inspect the data

Before we do any knowledge engineering, let's look at what our data engineering layer produces. Each function in `data_engineering.py` reads from a different source format and returns a Polars DataFrame with **URIs** ready for graph serialisation.

**Important**: the DataFrames below look clean and uniform, but the source files are *not*. Each file uses its own column naming conventions — the harmonization happens inside `data_engineering.py`. We'll look at that in section 1b.

In [3]:
# Customers — from Parquet (Databricks / lakehouse pattern)
data.customers()

name,customer_iri,email,country,segment,signup_date
str,str,str,str,str,str
"""Noah Hansen""","""http://data.treehouse.example/C0001""","""customer1@example.com""","""Germany""","""SMB""","""2023-10-17"""
"""Ava Lindström""","""http://data.treehouse.example/C0002""","""customer2@example.com""","""Sweden""","""Enterprise""","""2022-08-17"""
"""Henrik Hansen""","""http://data.treehouse.example/C0003""","""customer3@example.com""","""Norway""","""Enterprise""","""2023-08-28"""
"""Lars De Vries""","""http://data.treehouse.example/C0004""","""customer4@example.com""","""Netherlands""","""Enterprise""","""2022-08-12"""
"""Sigrid Lindström""","""http://data.treehouse.example/C0005""","""customer5@example.com""","""Spain""","""Government""","""2023-07-29"""
…,…,…,…,…,…
"""Liam Olsen""","""http://data.treehouse.example/C0021""","""customer21@example.com""","""UK""","""Government""","""2024-04-21"""
"""Olivia Virtanen""","""http://data.treehouse.example/C0022""","""customer22@example.com""","""Netherlands""","""Startup""","""2022-10-02"""
"""Karin García""","""http://data.treehouse.example/C0023""","""customer23@example.com""","""USA""","""Government""","""2022-08-06"""


In [4]:
# Orders — from CSV (classic flat-file)
data.orders()

order_id,order_iri,customer_iri,product_iri,quantity,total_amount,order_date,status
str,str,str,str,i64,f64,str,str
"""ORD-00001""","""http://data.treehouse.example/ORD-00001""","""http://data.treehouse.example/C0012""","""http://data.treehouse.example/P0004""",3,1569.72,"""2023-04-04""","""completed"""
"""ORD-00002""","""http://data.treehouse.example/ORD-00002""","""http://data.treehouse.example/C0004""","""http://data.treehouse.example/P0003""",3,2391.96,"""2024-03-08""","""completed"""
"""ORD-00003""","""http://data.treehouse.example/ORD-00003""","""http://data.treehouse.example/C0013""","""http://data.treehouse.example/P0007""",10,9952.4,"""2024-06-25""","""returned"""
"""ORD-00004""","""http://data.treehouse.example/ORD-00004""","""http://data.treehouse.example/C0018""","""http://data.treehouse.example/P0014""",1,688.87,"""2023-04-28""","""returned"""
"""ORD-00005""","""http://data.treehouse.example/ORD-00005""","""http://data.treehouse.example/C0025""","""http://data.treehouse.example/P0011""",6,823.26,"""2024-03-21""","""shipped"""
…,…,…,…,…,…,…,…
"""ORD-00056""","""http://data.treehouse.example/ORD-00056""","""http://data.treehouse.example/C0025""","""http://data.treehouse.example/P0005""",6,3907.44,"""2024-02-14""","""returned"""
"""ORD-00057""","""http://data.treehouse.example/ORD-00057""","""http://data.treehouse.example/C0001""","""http://data.treehouse.example/P0002""",5,1011.05,"""2023-09-29""","""completed"""
"""ORD-00058""","""http://data.treehouse.example/ORD-00058""","""http://data.treehouse.example/C0004""","""http://data.treehouse.example/P0010""",7,2550.24,"""2023-11-18""","""cancelled"""


In [5]:
# Products — from Excel (.xlsx, business-managed)
data.products()

product_name,product_iri,category,unit_price,stock_quantity
str,str,str,f64,i64
"""CloudSync Platform""","""http://data.treehouse.example/P0001""","""Software""",382.29,35
"""DataVault Pro""","""http://data.treehouse.example/P0002""","""Hardware""",949.41,471
"""NetGuard Firewall""","""http://data.treehouse.example/P0003""","""Services""",349.29,160
"""CodeCamp Workshop""","""http://data.treehouse.example/P0004""","""Training""",672.49,63
"""PrimeSupport Plan""","""http://data.treehouse.example/P0005""","""Support""",727.16,153
…,…,…,…,…
"""EdgeCompute Node""","""http://data.treehouse.example/P0011""","""Software""",686.0,462
"""InsightQL Engine""","""http://data.treehouse.example/P0012""","""Hardware""",197.81,291
"""SmartRouter X1""","""http://data.treehouse.example/P0013""","""Services""",320.92,280


In [6]:
# Support tickets — via SQL endpoint (Polars SQLContext)
data.support_tickets()

ticket_id,ticket_iri,customer_iri,issue_type,priority,ticket_status,created_date
str,str,str,str,str,str,str
"""TKT-00007""","""http://data.treehouse.example/TKT-00007""","""http://data.treehouse.example/C0023""","""feature_request""","""critical""","""closed""","""2024-12-22"""
"""TKT-00002""","""http://data.treehouse.example/TKT-00002""","""http://data.treehouse.example/C0015""","""technical""","""critical""","""in_progress""","""2024-12-11"""
"""TKT-00011""","""http://data.treehouse.example/TKT-00011""","""http://data.treehouse.example/C0015""","""bug_report""","""high""","""closed""","""2024-12-07"""
"""TKT-00031""","""http://data.treehouse.example/TKT-00031""","""http://data.treehouse.example/C0008""","""billing""","""low""","""open""","""2024-11-28"""
"""TKT-00025""","""http://data.treehouse.example/TKT-00025""","""http://data.treehouse.example/C0005""","""bug_report""","""low""","""closed""","""2024-11-14"""
…,…,…,…,…,…,…
"""TKT-00008""","""http://data.treehouse.example/TKT-00008""","""http://data.treehouse.example/C0008""","""billing""","""low""","""closed""","""2024-03-16"""
"""TKT-00003""","""http://data.treehouse.example/TKT-00003""","""http://data.treehouse.example/C0022""","""onboarding""","""high""","""open""","""2024-02-13"""
"""TKT-00006""","""http://data.treehouse.example/TKT-00006""","""http://data.treehouse.example/C0020""","""feature_request""","""critical""","""in_progress""","""2024-02-07"""


Notice how every DataFrame has a `*_iri` column — these will become the **subjects** (the identities) of our RDF triples. Columns that reference other entities (like `customer_iri` in orders) will become **object property links** — the edges in the graph.

This is the key insight from Part 1 of the blog series: *the data engineering produces DataFrames; the knowledge engineering turns them into a graph.*

---
## 1b. Entity harmonization — the mapping magic

The DataFrames above look clean. But the raw source files tell a different story. Let's peek at the **actual column names** before harmonization:

In [7]:
import polars as pl

# Read each file RAW — no harmonization, just the column names as-is
raw_customers  = pl.read_parquet("data/customers.parquet")
raw_orders     = pl.read_csv("data/orders.csv")
raw_products   = pl.read_excel("data/products.xlsx")
raw_tickets    = pl.read_csv("data/support_tickets.csv")

print("customers.parquet :", raw_customers.columns)
print("orders.csv        :", raw_orders.columns)
print("products.xlsx     :", raw_products.columns)
print("support_tickets   :", raw_tickets.columns)

customers.parquet : ['customer_id', 'full_name', 'email', 'country', 'segment', 'registered_at']
orders.csv        : ['order_id', 'cust_ref', 'prod_code', 'quantity', 'total_amount', 'order_date', 'order_status']
products.xlsx     : ['sku', 'name', 'category', 'price', 'in_stock']
support_tickets   : ['ticket_id', 'client_id', 'type', 'severity', 'state', 'date_created']


See the problem? The customer foreign key has **three different names** across three systems:

| Source | Column name | Refers to |
|--------|------------|-----------|
| customers.parquet | `customer_id` | The customer's own ID |
| orders.csv | `cust_ref` | Same customer, different name |
| support_tickets.csv | `client_id` | Same customer, yet another name |

And it gets worse. The column `name` in products.xlsx is a **product name**, while `full_name` in customers.parquet is a **person's name**. Column names alone don't carry semantics.

This is a core challenge: *"How do you know two talked about entities are the same?"* In LPG or RDF, if you're not careful, you end up with different nodes for the same real-world entity, just because the source systems use different identifiers or naming conventions.

The solution in a knowledge graph pipeline:
1. **Harmonize column names** to a shared vocabulary (rename `cust_ref` → `customer_id`, `sku` -> `product_id`, etc.)
2. **Construct IRIs** from the harmonized identifiers: the IRI becomes the single, global identity for that entity
3. **The ontology provides meaning** it tells you that `:email` on a `Customer` is not the same kind of thing as `name` on a `Product`, even if both happen to be strings

`data_engineering.py` has a helper function that shows the full mapping table:

In [8]:
# The full column harmonization map — source names → shared vocabulary
data.column_mapping_table()

source_file,source_column,harmonised_to,why
str,str,str,str
"""customers.parquet""","""full_name""","""name""","""OTTR template expects 'name'"""
"""customers.parquet""","""registered_at""","""signup_date""","""Normalise date naming"""
"""orders.csv""","""cust_ref""","""customer_id""","""Align with customer master PK"""
"""orders.csv""","""prod_code""","""product_id""","""Align with product catalog PK"""
"""orders.csv""","""order_status""","""status""","""Match OTTR template field"""
…,…,…,…
"""support_tickets.csv""","""client_id""","""customer_id""","""Align with customer master PK"""
"""support_tickets.csv""","""type""","""issue_type""","""Qualify as issue_type"""
"""support_tickets.csv""","""severity""","""priority""","""Align with SKOS priority scheme"""


This harmonization step is where data engineering meets knowledge engineering. Each `rename()` call in `data_engineering.py` is a deliberate alignment decision; mapping from the source system's vocabulary to the shared vocabulary defined by our OTTR templates and ontology.

Let's verify that the harmonization actually works. That the same customer gets the same IRI across all three sources:

In [9]:
# Pick a customer that appears in all three sources and check the IRIs match
sample_customer = "C0005"

customer_iri = data.customers().filter(
    pl.col("customer_iri").str.ends_with(sample_customer)
)["customer_iri"][0]

order_iri = data.orders().filter(
    pl.col("customer_iri").str.ends_with(sample_customer)
)["customer_iri"][0]

ticket_iri = data.support_tickets().filter(
    pl.col("customer_iri").str.ends_with(sample_customer)
)["customer_iri"][0]

print(f"Customer master (customer_id={sample_customer}):  {customer_iri}")
print(f"Order system    (cust_ref={sample_customer}):     {order_iri}")
print(f"Ticket system   (client_id={sample_customer}):    {ticket_iri}")
print(f"\nAll three match: {customer_iri == order_iri == ticket_iri}")

Customer master (customer_id=C0005):  http://data.treehouse.example/C0005
Order system    (cust_ref=C0005):     http://data.treehouse.example/C0005
Ticket system   (client_id=C0005):    http://data.treehouse.example/C0005

All three match: True


The IRI `http://data.treehouse.example/C0012` is the **single identity** for that customer — regardless of whether the source system called the column `customer_id`, `cust_ref`, or `client_id`. Once data enters the graph through these harmonized IRIs, all relationships connect automatically. No JOINs needed.

This is the answer to the entity resolution question: in a knowledge graph, identity is an **IRI**, not a column name. The data engineering layer constructs that IRI consistently across sources, and the ontology gives it meaning.

---
## 2. Initialise the Model and load OTTR templates

A `Model` is maplib's in-memory knowledge graph. We load [OTTR templates](https://ottr.xyz/) that define *how* each DataFrame maps to RDF triples. Meaning which columns become IRIs, which become typed literals, and what predicates connect them.

In [10]:
m = Model()

# Load the OTTR template definitions
with open("tpl/tpl.ttl", "r") as f:
    m.add_template(f.read())

print("Model initialised with OTTR templates.")

Model initialised with OTTR templates.


### What do the templates look like?

Each template declares a **signature** (the column names and their XSD types) and a **body** (the triples to generate). For example, the Customer template:

```turtle
tpl:Customer[
    xsd:string ?name,
    ! ottr:IRI ?customer_iri,   # subject IRI, '!' for mandatory
    xsd:string ?email,
    xsd:string ?country,
    xsd:string ?segment,
    xsd:string ?signup_date
] :: {
    ottr:Triple(?customer_iri, rdf:type, :Customer),
    ottr:Triple(?customer_iri, rdfs:label, ?name),
    ottr:Triple(?customer_iri, :email, ?email),
    ...
} .
```

The template maps each row in the DataFrame to a set of triples. The column names in the DataFrame must match the variable names in the template signature.

---
## 3. Map data frames to RDF

Now we serialise each DataFrame into the knowledge graph using the IRI (e.g. `data.ns_tpl + "Customer"`) that corresponds to the OTTR template. This is where **data becomes graph**.

In [11]:
m.map(data.ns_tpl + "Customer", data.customers())
m.map(data.ns_tpl + "Order", data.orders())
m.map(data.ns_tpl + "Product", data.products())
m.map(data.ns_tpl + "SupportTicket", data.support_tickets())

print_count("mapping all four data sources", m)

Graph size after mapping all four data sources :  985


Let's verify by running a quick SPARQL query. List all customers with their country and segment:

In [12]:
m.query("""
    PREFIX : <http://data.treehouse.example/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?name ?country ?segment
    WHERE {
        ?c a :Customer ;
           rdfs:label ?name ;
           :country ?country ;
           :segment ?segment .
    }
    ORDER BY ?name
""")

name,country,segment
str,str,str
"""Astrid Johansson""","""UK""","""Startup"""
"""Ava Lindström""","""Sweden""","""Enterprise"""
"""Ava Virtanen""","""UK""","""Enterprise"""
"""Bjorn Virtanen""","""Germany""","""Enterprise"""
"""Elise Dubois""","""Netherlands""","""Government"""
…,…,…
"""Noah Hansen""","""Germany""","""SMB"""
"""Noah Schmidt""","""USA""","""Enterprise"""
"""Olivia Virtanen""","""Netherlands""","""Startup"""


And a cross-entity query: which customer placed which order, and for how much? Notice how this traverses the `:placedBy` relationship that connects orders to customers. 

**No JOINs needed!** The relationship is a first-class citizen in the graph. <3

In [13]:
m.query("""
    PREFIX : <http://data.treehouse.example/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?customer_name ?order_id ?amount
    WHERE {
        ?order a :Order ;
               rdfs:label ?order_id ;
               :placedBy ?customer ;
               :totalAmount ?amount .
        ?customer rdfs:label ?customer_name .
    }
    ORDER BY ?customer_name
    LIMIT 10
""")

customer_name,order_id,amount
str,str,f64
"""Astrid Johansson""","""ORD-00060""",214.48
"""Astrid Johansson""","""ORD-00042""",2333.1
"""Astrid Johansson""","""ORD-00024""",3509.0
"""Astrid Johansson""","""ORD-00035""",1563.18
"""Ava Lindström""","""ORD-00022""",4833.42
"""Ava Lindström""","""ORD-00032""",1709.6
"""Ava Virtanen""","""ORD-00010""",3522.68
"""Bjorn Virtanen""","""ORD-00013""",4394.4
"""Bjorn Virtanen""","""ORD-00038""",2802.78


---
## 4. Merge the ontology

The ontology (`ttl/ontology.ttl`) is the **knowledge layer**. It defines what a Customer *is*, what relationships are valid, and how our classes connect to external standards like [FIBO](https://spec.edmcouncil.org/fibo/) (Financial Industry Business Ontology).

Since everything in a knowledge graph is triples, merging the ontology is just reading more triples into the same model.

In [14]:
m.read("ttl/ontology.ttl")
print_count("merge with ontology", m)

Graph size after merge with ontology :  1118


Let's verify the FIBO alignment — our classes are now linked to standardised financial industry concepts:

In [15]:
m.query("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?local_class ?fibo_superclass
    WHERE {
        ?local_class rdfs:subClassOf ?fibo_superclass .
        FILTER(STRSTARTS(STR(?fibo_superclass), "https://spec.edmcouncil.org/fibo"))
    }
""")

local_class,fibo_superclass
str,str
"""<http://data.treehouse.example/Customer>""","""<https://spec.edmcouncil.org/fibo/ontology/FND/Parties/Roles/AgentInRole>"""
"""<http://data.treehouse.example/Order>""","""<https://spec.edmcouncil.org/fibo/ontology/FND/Agreements/Agreements/MutualCommitment>"""
"""<http://data.treehouse.example/Product>""","""<https://spec.edmcouncil.org/fibo/ontology/FND/ProductsAndServices/ProductsAndServices/Product>"""


---
## 5. Merge SKOS vocabularies

[SKOS](https://www.w3.org/TR/skos-reference/) (Simple Knowledge Organisation System) provides **controlled vocabularies**. Proper concept definitions for things like product categories, customer segments, issue types and priorities.

Instead of just having a string `"Software"` hanging off a product, we get a SKOS Concept with a definition, related concepts, and a place in a hierarchy.

In [16]:
m.read("ttl/vocab.ttl")
print_count("merge with SKOS vocabularies", m)

Graph size after merge with SKOS vocabularies :  1207


Let's peek at the concept schemes and their concepts:

In [17]:
m.query("""
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX dct: <http://purl.org/dc/terms/>

    SELECT ?scheme ?concept_label ?definition
    WHERE {
        ?scheme a skos:ConceptScheme ;
                skos:prefLabel ?scheme_name .
        ?concept skos:inScheme ?scheme ;
                 skos:prefLabel ?concept_label .
        OPTIONAL { ?concept skos:definition ?definition . }
    }
    ORDER BY ?scheme ?concept_label
""")

scheme,concept_label,definition
str,str,str
"""<http://data.treehouse.example/CustomerSegmentScheme>""","""""Enterprise""@en""","""""Large organisations with complex needs and high-value contracts.""@en"""
"""<http://data.treehouse.example/CustomerSegmentScheme>""","""""Government""@en""","""""Public sector organisations and agencies.""@en"""
"""<http://data.treehouse.example/CustomerSegmentScheme>""","""""SMB""@en""","""""Small and medium-sized businesses.""@en"""
"""<http://data.treehouse.example/CustomerSegmentScheme>""","""""Startup""@en""","""""Early-stage companies with growth potential.""@en"""
"""<http://data.treehouse.example/IssueTypeScheme>""","""""billing""@en""","""""Issues related to invoicing, payments and account charges.""@en"""
…,…,…
"""<http://data.treehouse.example/ProductCategoryScheme>""","""""Hardware""@en""","""""Physical computing equipment such as GPUs, routers and compute nodes.""@en"""
"""<http://data.treehouse.example/ProductCategoryScheme>""","""""Services""@en""","""""Professional services including ETL pipelines, consulting and managed offerings.""@en"""
"""<http://data.treehouse.example/ProductCategoryScheme>""","""""Software""@en""","""""Digital products including platforms, tools and engines.""@en"""


---
## 6. Merge DCAT catalog

[DCAT](https://www.w3.org/TR/vocab-dcat-3/) (Data Catalog Vocabulary) describes the **data sources themselves**. Their format, access URL, and descriptions. This is metadata that travels *with* the knowledge graph.

In [18]:
m.read("ttl/catalog.ttl")
print_count("merge with DCAT catalog", m)

Graph size after merge with DCAT catalog :  1250


In [19]:
m.query("""
    PREFIX dcat: <http://www.w3.org/ns/dcat#>
    PREFIX dct: <http://purl.org/dc/terms/>

    SELECT ?dataset_title ?format ?media_type
    WHERE {
        ?ds a dcat:Dataset ;
            dct:title ?dataset_title ;
            dcat:distribution ?dist .
        ?dist dct:format ?format .
        OPTIONAL { ?dist dcat:mediaType ?media_type . }
    }
""")

dataset_title,format,media_type
str,str,str
"""""Customer Master Data""@en""","""Parquet""","""application/vnd.apache.parquet"""
"""""Order Transaction Data""@en""","""CSV""","""text/csv"""
"""""Product Catalog""@en""","""Excel (XLSX)""","""application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"""
"""""Support Ticket Data""@en""","""CSV (accessed via SQL)""","""text/csv"""


---
## 7. Link instances to SKOS concepts

Right now, our products have a plain string `:category "Software"` and our customers have `:segment "Enterprise"`. The SKOS vocabulary has proper concepts with those labels.

We bridge the two using INSERT queries that **match on `skos:prefLabel`** and create typed object property links (`:hasCategory`, `:hasSegment`, `:hasIssueType`, `:hasPriority`).

After this step, you can navigate from any product to its SKOS concept, read its definition, and discover related concepts — all through the graph.

In [20]:
skos_queries = [
    "queries/link_categories_to_skos.rq",
    "queries/link_segments_to_skos.rq",
    "queries/link_issues_to_skos.rq",
    "queries/link_priorities_to_skos.rq",
]

for qf in skos_queries:
    with open(qf, "r") as f:
        m.update(f.read())
    print(f"  ✓ {qf}")

print_count("SKOS concept linking", m)

  ✓ queries/link_categories_to_skos.rq
  ✓ queries/link_segments_to_skos.rq
  ✓ queries/link_issues_to_skos.rq
  ✓ queries/link_priorities_to_skos.rq
Graph size after SKOS concept linking :  1370


Let's verify! Products should now have both a string `:category` *and* a typed `:hasCategory` link to a SKOS concept with a definition:

In [21]:
m.query("""
    PREFIX : <http://data.treehouse.example/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT ?product_name ?category_label ?definition
    WHERE {
        ?product a :Product ;
                 rdfs:label ?product_name ;
                 :hasCategory ?concept .
        ?concept skos:prefLabel ?category_label .
        OPTIONAL { ?concept skos:definition ?definition . }
    }
    ORDER BY ?category_label ?product_name
""")

product_name,category_label,definition
str,str,str
"""DataVault Pro""","""""Hardware""@en""","""""Physical computing equipment such as GPUs, routers and compute nodes.""@en"""
"""InsightQL Engine""","""""Hardware""@en""","""""Physical computing equipment such as GPUs, routers and compute nodes.""@en"""
"""StreamFlow ETL""","""""Hardware""@en""","""""Physical computing equipment such as GPUs, routers and compute nodes.""@en"""
"""NetGuard Firewall""","""""Services""@en""","""""Professional services including ETL pipelines, consulting and managed offerings.""@en"""
"""SecureID Module""","""""Services""@en""","""""Professional services including ETL pipelines, consulting and managed offerings.""@en"""
…,…,…
"""Enterprise SLA Gold""","""""Support""@en""","""""Customer support plans including helpdesk and SLA-based offerings.""@en"""
"""PrimeSupport Plan""","""""Support""@en""","""""Customer support plans including helpdesk and SLA-based offerings.""@en"""
"""CodeCamp Workshop""","""""Training""@en""","""""Educational offerings such as workshops, bootcamps and courses.""@en"""


---
## 8. Inference--deriving facts that don't exist in the source data

This is where the knowledge graph starts doing things SQL simply cannot. The Datalog rules in `ttl/rule.dlog` derive new triples from existing ones:

1. **Inverse relationships**: `:hasOrder` and `:hasTicket` (from `:placedBy` and `:raisedBy`)
2. **Transitive purchasing**: if a customer placed an order containing a product → `:purchased`
3. **Product classification**: `:PremiumProduct` (price > 500) and `:BudgetProduct` (price ≤ 100)

After inference, you can query these derived facts as if they were always there.

> **Note**: `m.infer()` requires the maplib enterprise engine. If you're using the open-source version, the cell below will error--that's expected. The rest of the notebook still works; you just won't have the inferred triples. You can always get a free license for personal usage. 

In [22]:
with open("ttl/rule.dlog", "r") as f:
    rules = f.read()

print("Datalog rules:")
print(rules)

Datalog rules:
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX : <http://data.treehouse.example/>

# Inverse of placedBy is hasOrder — if an order is placed by a customer, the customer has that order
[?customer, :hasOrder, ?order] :- [?order, :placedBy, ?customer] .

# Inverse of raisedBy is hasTicket — if a ticket is raised by a customer, the customer has that ticket
[?customer, :hasTicket, ?ticket] :- [?ticket, :raisedBy, ?customer] .

# If a customer placed an order that contains a product, the customer purchased that product
[?customer, :purchased, ?product] :- [?order, :placedBy, ?customer], [?order, :contains, ?product] .

# Classify products as Premium (unit price > 500) or Budget (unit price <= 100)
[?product, rdf:type, :PremiumProduct] :-
  [?product, rdf:type, :Product],
  [?product, :unitPrice, ?price] ,
  FILTER(?price > 500) .

[?product, rdf:type, :BudgetProduct] :-
  [?product, rdf:type, :Product],
  [?product, :unitPrice, ?price] ,
  FILTER(?price <= 10

In [23]:
try:
    m.infer(rules)
    print_count("inference", m)
except BaseException as e:
    print(f"Inference requires maplib enterprise: {type(e).__name__}")
    print("Continuing without inferred triples — the rest of the notebook still works.")

Graph size after inference :  1534


If inference ran, we can now query the *derived* `:purchased` relationship — a fact that does not exist in any source file:

In [24]:
df_purchased = m.query("""
    PREFIX : <http://data.treehouse.example/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?customer_name ?product_name
    WHERE {
        ?customer :purchased ?product .
        ?customer rdfs:label ?customer_name .
        ?product  rdfs:label ?product_name .
    }
    ORDER BY ?customer_name ?product_name
""")

if len(df_purchased) > 0:
    print(f"{len(df_purchased)} customer→product purchase relationships inferred!")
    df_purchased
else:
    print("No :purchased triples found. Inference likely didn't run (see note above).")

56 customer→product purchase relationships inferred!


---
## 9. SHACL Validation--trusting the data

[SHACL](https://www.w3.org/TR/shacl/) (Shapes Constraint Language) lets us define **what valid data looks like** declaratively, as triples. The shapes in `ttl/sh.ttl` say things like:

- Every Customer must have an `:email` (string) and a `:country` (string)
- Every Order must have a `:placedBy` link to a `:Customer`

When we validate, we get a machine-readable report, triples describing exactly what went wrong.

> **Note**: `m.validate()` requires the maplib enterprise engine. If you're using the open-source version, the cell below will error--that's expected. The rest of the notebook still works; you just won't have the inferred triples. You can always get a free license for personal usage. 

In [28]:
m.read("ttl/sh.ttl", graph=data.ns_sh)

report = m.validate(shape_graph=data.ns_sh)

df_report = report.results()
print(f"Validation results: {len(df_report)} row(s)")
df_report

Validation results: 0 row(s)


focus_node,source_shape,source_constraint_component,result_path,result_severity,message,value,conforms
str,str,str,str,str,str,str,bool


If the report is empty (0 rows), our data **conforms** to the shapes. If there are violations, each row tells you exactly which node (`focusNode`), which property (`resultPath`), and what went wrong (`resultMessage`).

We can also query the validation report as a graph:

In [31]:
with open("queries/focus_node_violations.rq", "r") as f:
    focus_query = f.read()

violations = m.query(focus_query)
if len(violations) > 0:
    print("Focus nodes with violations:")
    print(violations)
else:
    print("No violations found. The data conforms to all SHACL shapes.")

No violations found. The data conforms to all SHACL shapes.


---
## 10. SPARQL Competency Questions

The `queries/` folder contains 8 competency questions that showcase what SPARQL + a knowledge graph can do that SQL alone cannot. Let's run a few of them.

### CQ4: Schema introspection

*What classes exist in this graph, what properties belong to each, and how many instances?*

In a knowledge graph, the schema **is** data. Stored as triples just like everything else. SQL cannot mix `information_schema` with instance queries in a single statement.

In [ ]:
with open("queries/cq4.rq", "r") as f:
    cq4 = f.read()

m.query(cq4)

### CQ3: Customers with orders but no tickets (FILTER NOT EXISTS)

*Which customers have placed orders but never raised a support ticket?*

SPARQL's `FILTER NOT EXISTS` is elegant negation--it reads like English. The SQL equivalent needs a `LEFT JOIN ... WHERE ... IS NULL`.

In [ ]:
with open("queries/cq3.rq", "r") as f:
    cq3 = f.read()

m.query(cq3)

### CQ5: Customer 360° view

*For a given customer, return everything related: orders, products, product classifications, tickets--all in one query.*

SPARQL `OPTIONAL` blocks are independently composable. In SQL, the equivalent multi-LEFT-JOIN produces a Cartesian explosion (orders × tickets).

---
## Summary

In this notebook we walked through the complete knowledge engineering pipeline:

| Step | What happened | Key concept |
|------|--------------|-------------|
| 1 | Inspected data frames from 4 sources | Data engineering produces URIs |
| 1b | Entity harmonization across sources | Column names → shared vocabulary → IRIs |
| 2 | Loaded OTTR templates | Templates are your mapping contract |
| 3 | Mapped data frames → RDF | Data becomes graph |
| 4 | Merged the ontology | Schema + data = knowledge graph |
| 5 | Merged SKOS vocabularies | Controlled vocabularies for terms |
| 6 | Merged DCAT catalog | Metadata about the sources |
| 7 | Linked instances to SKOS concepts | Bridging strings to meaning |
| 8 | Ran Datalog inference | Deriving facts not in the source data |
| 9 | Validated with SHACL | Declarative data quality |
| 10 | Ran SPARQL competency questions | Asking questions SQL can't |
| 11 | Wrote the graph to Turtle | Everything in one file |

The full set of 8 competency questions is in the `queries/` folder, each with a SQL equivalent in the comments. Try them out!

### CQ1: Inferred purchases (requires inference)

*Which products has each customer purchased?*

The `:purchased` relationship was **inferred** by the Datalog rule. It doesn't exist in any source file. In SPARQL we just query it. In SQL, you'd need a JOIN through the orders table.

In [ ]:
with open("queries/cq1.rq", "r") as f:
    cq1 = f.read()

df_cq1 = m.query(cq1)
if len(df_cq1) > 0:
    display(df_cq1)
else:
    print("No results — this CQ requires inferred :purchased triples (see section 8).")

---
## 11. Write the graph to file

Finally, we can serialise the entire knowledge graph to a Turtle file. This file contains *everything*; the instance data, the ontology, the SKOS vocabularies, the DCAT catalog, and (if inference ran) the derived triples.

In [ ]:
p = {"": "http://data.treehouse.example/"}
m.write("ttl/out.ttl", format="turtle", prefixes=p)

# Check the output size
import os
size = os.path.getsize("ttl/out.ttl")
print(f"Written to ttl/out.ttl ({size:,} bytes)")
print_count("final graph", m)

---
## 12. Explore interactively

maplib ships with a built-in web UI for exploring the graph. Uncomment and run the cell below to launch it. It will open a browser window where you can navigate nodes and expand relationships.

In [ ]:
# Uncomment to launch the interactive explorer:
# m.explore()

---
## Summary

In this notebook we walked through the complete knowledge engineering pipeline:

| Step | What happened | Key concept |
|------|--------------|-------------|
| 1 | Inspected data frames from 4 sources | Data engineering produces IRIs |
| 2 | Loaded OTTR templates | Templates are your mapping contract |
| 3 | Mapped data frames → RDF | Data becomes graph |
| 4 | Merged the ontology | Schema + data = knowledge graph |
| 5 | Merged SKOS vocabularies | Controlled vocabularies for terms |
| 6 | Merged DCAT catalog | Metadata about the sources |
| 7 | Linked instances to SKOS concepts | Bridging strings to meaning |
| 8 | Ran Datalog inference | Deriving facts not in the source data |
| 9 | Validated with SHACL | Declarative data quality |
| 10 | Ran SPARQL competency questions | Asking questions SQL can't |
| 11 | Wrote the graph to Turtle | Everything in one file |

The full set of 8 competency questions is in the `queries/` folder — each with a SQL equivalent in the comments. Try them out!